In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [2]:
import warnings
import os
import logging
# 경고 제거
warnings.filterwarnings('ignore')

# transformers 로깅 레벨 조정
logging.getLogger("transformers").setLevel(logging.ERROR)

# Hugging Face symlink 경고 제거
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# from transformers import pipeline, logging as hf_logging
# hf_logging.set_verbosity_error()

<b><font size="6" color="red">ch.1 허깅페이스</font></b>
- Inference API 이용 : 모델의 결과를 server에서
- pipeline() 이용 : 모델을 다운로드 받아 모델의 결과를 local에서
    * raw text -> tokenizer -> model -> [0.11, 0.55, 0.xx, ~] logits값으로 prediction 결과 출력
```
허깅페이스 transformers에서 지원하는 task
'sentiment-analysis' : 'text-classification'의 별칭(감정분석 전용으로 사용
'text-classification' : 감정분석, 뉴스분류, 리뷰 분류 등 일반적인 문장 분류
'zore-shot-classification' : 레이블을 학습 없이 주어진 후보군 중에서 분류
'token-classification' : 개체명 인식(NER : Named Entity Recognition) 등 단위 라벨링
'ner' : 'token-classification'의 별칭
'fill-mask' : 빈칸 채우기
'text-generation' : 텍스트 생성(GPT류 모델에 사용)
'text2text-generation' : 번역, 요약 등 입력 -> 출력 변환
'translation' : 번역
'summarization' : 텍스트 요약
'qustion-answering' : 주어진 context를 보고 질문에 답하기.
'image-to-text' : 그림을 설명
'image-classification' : 이미지분류
```

# 1. 텍스트 기반 감정분석(긍정/부정)
- C:\사용자\내컴퓨터명\.cache\huggingface\hue 모델 다운로드

In [4]:
from transformers import pipeline
classifier = pipeline(task = 'sentiment-analysis') 
classifier("I've been waiting for a HuggingFace course my whole life.")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9598049521446228}]

In [13]:
classifier = pipeline(task = 'text-classification',
                     model = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english')
# 감정분석시 내용이 많으면 list로
classifier([
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate you"
])

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9598049521446228},
 {'label': 'NEGATIVE', 'score': 0.9991129040718079}]

In [14]:
classifier(['이 영화는 최고였어요. 감동적이고 연기가 대단해',
           "This movie was the best. It's touching, and the acting is amazing"])

[{'label': 'POSITIVE', 'score': 0.880291759967804},
 {'label': 'POSITIVE', 'score': 0.9998821020126343}]

In [8]:
classifier('이 물건 정말 사고 싶어요')

[{'label': 'POSITIVE', 'score': 0.8577604293823242}]

In [9]:
classifier(['I like you','I hate you', '나 너가 싫어', '힘들어요'
           ])

[{'label': 'POSITIVE', 'score': 0.9998695850372314},
 {'label': 'NEGATIVE', 'score': 0.9991129040718079},
 {'label': 'NEGATIVE', 'score': 0.599323034286499},
 {'label': 'POSITIVE', 'score': 0.8669533729553223}]

In [16]:
classifier = pipeline("text-classification", model="matthewburke/korean_sentiment")
texts = ['나는 너가 좋아', '당신이 싫어요', '힘들어요', '오늘 기분이 최고야']
result = classifier(texts)

Device set to use cpu


In [28]:
for text, result in zip(texts, classifier(texts)):
    label = '긍정' if result['label'] == 'LABEL_1' else '부정'
    print(f'{text} = > {label} : {result["score"]:.4f}')

나는 너가 좋아 = > 긍정 : 0.9558
당신이 싫어요 = > 부정 : 0.9093
힘들어요 = > 부정 : 0.9140
오늘 기분이 최고야 = > 긍정 : 0.9714


# 2. 제로샷분류(Zero-shot분류)
- 기계학습 및 자연어처리에서 각 개별 작업에 대한 특정 교육없이 작업을 수행할 수 있는 모형(비지도학습)

In [30]:
classifier = pipeline("zero-shot-classification",
                     model = 'facebook/bart-large-mnli')
classifier(
    'I haver a problem with my iphone that need  to be resolved asap!',
    candidate_labels=['urgent', 'not urgent', 'phone', 'tablet', 'computer']
)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


{'sequence': 'I haver a problem with my iphone that need  to be resolved asap!',
 'labels': ['phone', 'urgent', 'computer', 'not urgent', 'tablet'],
 'scores': [0.49313515424728394,
  0.49021443724632263,
  0.010512522421777248,
  0.0032299510203301907,
  0.002907864283770323]}

In [31]:
sequence_to_classifier = 'One day I well see the world'
candidate_labels = ['traval', 'cooking', 'dancing']
classifier(sequence_to_classifier, candidate_labels)

{'sequence': 'One day I well see the world',
 'labels': ['traval', 'cooking', 'dancing'],
 'scores': [0.7960318922996521, 0.10210832208395004, 0.10185977071523666]}

# 3. text 생성

In [36]:
from transformers import pipeline, set_seed
# set_seed(2)
generation = pipeline('text-generation') # gpt2(텍스트생성) gpt3부터는 허깅페이스에 없음
generation(
    'in this course. We will teach you how to',
    pad_token_id = generation.tokenizer.eos_token_id
)


No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


[{'generated_text': "in this course. We will teach you how to create and use virtual objects in Python 3.5.\n\nThe course is designed as a primer for learning Python 3.5, so let's get started.\n\nThe Introduction\n\nThe Introduction is the third series of tutorials that cover the basic concepts of virtual objects in Python. The first part of the course is a collection of exercises for beginners, including exercises that show you how to create a virtual object with Python 3. It's a great learning tool for students who want to learn from the experience of being an experienced developer.\n\nThe second part of the course covers all the basics of real-world virtual objects, including the basics of creating virtual objects with Python 3. The third part of the course includes a lot of examples that will help you identify the most common issues with virtual objects.\n\nThe 3.5 course will be divided into three parts, each covering a different topic. The first part covers the basics of virtual 

In [39]:
result = generation(
    'in this course. We will teach you how to',
    pad_token_id = generation.tokenizer.eos_token_id
)

print(result[0]['generated_text'])

in this course. We will teach you how to learn to code and how to learn a language. We will teach you how to learn to learn to code and how to learn a language.

We will teach you how to learn to code and how to learn a language. We will teach you how to learn to code and how to learn a language.

We will teach you how to learn to code and how to learn a language. We will teach you how to learn to code and how to learn a language.

We will teach you how to learn to code and how to learn a language. We will teach you how to learn to code and how to learn a language.

We will teach you how to learn to code and how to learn a language. We will teach you how to learn to code and how to learn a language.

We will teach you how to learn to code and how to learn a language. We will teach you how to learn to code and how to learn a language.

We will teach you how to learn to code and how to learn a language. We will teach you how to learn to code and how to learn a language.

We will teach yo

In [45]:
generation = pipeline('text-generation', 'skt/kogpt2-base-v2')
result = generation(
    '이 과정은 다음과 같은 방법을 알려드려요. ',
    pad_token_id = generation.tokenizer.eos_token_id,
    max_new_tokens = 100,     # 생성할 최대 길이(생성할 토큰 수)
    num_return_sequences = 1, # 생성할 문장 갯수
    do_sample = True,         # 다양한 샘플 사용 
    top_k = 50,               # top_k 샘플링(확률 높은 상위 50개 토큰만 사용)
    top_p = 0.95,             # 확률이 높은 순서대로 95%가 될 때까지의 단어들로만 후보로 사용
    temperature = 1.2,        # 창의성 조절(낮을수록 보수적)
    no_repeat_ngram_size = 2  # 반복 방지
)
print(result[0]['generated_text'])

Device set to use cpu


이 과정은 다음과 같은 방법을 알려드려요. 죠니가 원하는 걸 알면 그게 도움이 될 거예요."
이렇게 해서 쥬르니는 자신의 마음을 읽어낼 때 한 가지 분명한 교훈을 알게 되었습니다.
"그저 그렇다는 게 아니라, 왜 이렇게 힘들지 않았는지 이유를 설명할 수 있을 겁니다. 그 이유를 설명하면 그건 내가 원하는 것을 알 수 있다는 거죠."
쥬르의 질문에 딥브레인은 아무래도 괜찮다는 생각을 했어요.
'마음이 힘든 거야? 그냥 쉬면서 해봐야겠다'라고 생각하는


# 4. 마스크(빈칸) 채우기

In [47]:
unmasker = pipeline(task='fill-mask',
                   model = 'distilbert/distilroberta-base')
unmasker("I'm going to hospital and meet a <mask>")

Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


[{'score': 0.19275707006454468,
  'token': 3299,
  'token_str': ' doctor',
  'sequence': "I'm going to hospital and meet a doctor"},
 {'score': 0.06794589757919312,
  'token': 27321,
  'token_str': ' psychiatrist',
  'sequence': "I'm going to hospital and meet a psychiatrist"},
 {'score': 0.06435535103082657,
  'token': 16308,
  'token_str': ' surgeon',
  'sequence': "I'm going to hospital and meet a surgeon"},
 {'score': 0.0591287724673748,
  'token': 9008,
  'token_str': ' nurse',
  'sequence': "I'm going to hospital and meet a nurse"},
 {'score': 0.05705631151795387,
  'token': 1441,
  'token_str': ' friend',
  'sequence': "I'm going to hospital and meet a friend"}]

In [49]:
# unmasker('병원에 가서 <mask>를 만날 거에요')

In [50]:
unmasker("Hello, I'm a <mask> model.")

[{'score': 0.0629730075597763,
  'token': 265,
  'token_str': ' business',
  'sequence': "Hello, I'm a business model."},
 {'score': 0.038101598620414734,
  'token': 18150,
  'token_str': ' freelance',
  'sequence': "Hello, I'm a freelance model."},
 {'score': 0.03764132782816887,
  'token': 774,
  'token_str': ' role',
  'sequence': "Hello, I'm a role model."},
 {'score': 0.037326786667108536,
  'token': 2734,
  'token_str': ' fashion',
  'sequence': "Hello, I'm a fashion model."},
 {'score': 0.026023676618933678,
  'token': 24526,
  'token_str': ' Playboy',
  'sequence': "Hello, I'm a Playboy model."}]

In [52]:
unmasker('안녕하세요 나는 <mask>모델 입니다.')

[{'score': 0.4642501771450043,
  'token': 1437,
  'token_str': ' ',
  'sequence': '안녕하세요 나는 모델 입니다.'},
 {'score': 0.08251278102397919,
  'token': 12,
  'token_str': '-',
  'sequence': '안녕하세요 나는-모델 입니다.'},
 {'score': 0.04810582846403122,
  'token': 2,
  'token_str': '</s>',
  'sequence': '안녕하세요 나는모델 입니다.'},
 {'score': 0.03808755800127983,
  'token': 34437,
  'token_str': '~',
  'sequence': '안녕하세요 나는~모델 입니다.'},
 {'score': 0.037204477936029434,
  'token': 2383,
  'token_str': '–',
  'sequence': '안녕하세요 나는–모델 입니다.'}]

In [56]:
unmasker = pipeline(task='fill-mask',
                   model = 'google-bert/bert-base-uncased')
unmasker = ("Hello, I'm a [MASK] model")

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


## ※ InferenceAPI

In [60]:
from dotenv import load_dotenv
import os
load_dotenv()
# os.environ['HF_TOKEN']
# 허깅페이스 토큰을 READ 권한으로 생성하여 .env에 추가

True